In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q02/q02_rewrite/checkpoints/pre_cell_5.pickle

In [ ]:
%%cudf.pandas.profile
### cell 5 ###

# Cell 5 optimized
total = (
    nation[['N_NATIONKEY', 'N_NAME', 'N_REGIONKEY']]
    .merge(
        region[region['R_NAME'] == 'EUROPE'][['R_REGIONKEY']],
        left_on='N_REGIONKEY', right_on='R_REGIONKEY'
    )
    .merge(
        supplier[['S_SUPPKEY', 'S_NAME', 'S_ADDRESS', 'S_NATIONKEY', 'S_PHONE', 'S_ACCTBAL', 'S_COMMENT']],
        left_on='N_NATIONKEY', right_on='S_NATIONKEY'
    )
    .merge(
        partsupp[['PS_PARTKEY', 'PS_SUPPKEY', 'PS_SUPPLYCOST']],
        left_on='S_SUPPKEY', right_on='PS_SUPPKEY'
    )
    .merge(
        part[(part['P_SIZE'] == 15) & (part['P_TYPE'].str.endswith('BRASS'))][['P_PARTKEY', 'P_MFGR']],
        left_on='PS_PARTKEY', right_on='P_PARTKEY'
    )
    .assign(
        MIN_SUPPLYCOST=lambda df: df.groupby('P_PARTKEY')['PS_SUPPLYCOST'].transform('min')
    )
    .loc[lambda df: df['PS_SUPPLYCOST'] == df['MIN_SUPPLYCOST']]
    .drop(columns='MIN_SUPPLYCOST')[
        ['S_ACCTBAL', 'S_NAME', 'N_NAME', 'P_PARTKEY', 'P_MFGR', 'S_ADDRESS', 'S_PHONE', 'S_COMMENT']
    ]
    .sort_values(
        by=['S_ACCTBAL', 'N_NAME', 'S_NAME', 'P_PARTKEY'],
        ascending=[False, True, True, True]
    )
)
